# RFI development v1 — fine flags, separable DPSS computation

**Question.** Can we detect band-local bursts and 8/4/2-channel comb onsets promptly, recover quiet FM samples, and fit a smooth background without one matrix inversion per time row?

This is a notebook-only exploration, not a published flag product. It reproduces v0's final 20 minutes of span 13, using phase-C inputs **0 (ground), 4 (air), 04 (cross)**. The model targets the air auto. `flag == True` means **exclude**; the model and flags are separate outputs. Auto units are correlator counts, frequency MHz, time UTC seconds. The z-score retains v0's approximate radiometer normalization; it is not a calibrated Gaussian significance.

**Approach.** Keep v0's DPSS widths (50 ns, 1 mHz), but fit a tensor-product model with an arbitrary 2-D mask. Matrix-free conjugate gradients applies only small-basis time/frequency projections. Two small separable preconditioners replace thousands of row-dependent inversions. Positive-excess clipping initializes the model; band and comb evidence then expands the fitting exclusions before a final refit.

**Cheapest useful tests / acceptance.** Reproduce v0; verify the iterative solver against a dense weighted solve on a small problem; check every solve converges; measure FM retention and held-out residuals; inject known continuum, FM lines, bursts and all three comb spacings. Report first-integration detection, false flags, model error and runtime. Increased FM retention alone is not evidence of clean data. These tests must run before deciding whether to expand to other visits.

**Assumptions / uncertainties.** The continuum is stable enough for these DPSS widths over 20 minutes. RFI is usually positive auto excess. Negative residuals can indicate model failure or dropouts and are excluded separately. Frequencies above 235 MHz are explicitly outside the tested domain because of the low-count band edge. Thresholds are exploratory and declared below. No truth labels exist for the real FM gaps. Injection recovery is conditional on its idealized continuum and noise.

The engineering gate was explicitly waived by Aaron for this exploratory notebook. No package code, raw observations or curated products are changed.

In [ ]:
from pathlib import Path
import hashlib, inspect, json, subprocess, time, warnings
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from scipy.ndimage import median_filter, maximum_filter
from scipy.linalg import cho_factor, cho_solve
from scipy.sparse.linalg import LinearOperator, cg
from threadpoolctl import threadpool_limits
from hera_filters.dspec import dpss_operator, fit_solution_matrix
from eigsep_data import MetadataIndex
from IPython.display import display
%matplotlib widget

# Notebook execution uses this directory; resolve every input from here/parents.
NOTEBOOK_DIR = next(p / 'data-analysis/notebooks/arp/marjum-2026-07'
                    for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'data-analysis/notebooks/arp/marjum-2026-07/rfi_dev_v0.ipynb').exists())
ROOT = NOTEBOOK_DIR.parents[3]
CAMPAIGN = ROOT / 'marjum-2026-07'
# A reproducible thread limit prevents small BLAS products oversubscribing CPUs.
blas_limit = threadpool_limits(limits=1)
SPAN, MINUTES = 13, 20
BAND_MHZ = (35., 250.)             # Same load and baseline domain as v0.
TESTED_BAND_MHZ = (35., 235.)     # Upper edge remains flagged in v1.
REPORT_BANDS = {
    '50-88 (DTV-lo)': (50., 88.), '88-108 (FM)': (88., 108.),
    '108-137': (108., 136.5), '137-138 (Orbcomm)': (136.5, 138.5),
    '138-174': (138.5, 174.), '174-216 (DTV-hi)': (174., 216.),
    '216-235': (216., 235.),
}
P = dict(freq_halfwidth_s=50e-9, time_halfwidth_hz=1e-3,
         eigenvalue_cutoff=1e-12, coherence_cut=20., fit_clip=8.,
         point_cut=6., other_point_cut=8., negative_cut=8.,
         group_cut=6., group_cell_cut=2., group_fraction=.35,
         tile_channels=8, comb_cut=6., comb_tooth_cut=2.,
         comb_fraction=.10, comb_min_teeth=16, ridge=1e-6,
         cg_rtol=1e-8, cg_maxiter=200, robust_rounds=10,
         mask_change_tol=5e-4, time_guard=1, frequency_guard=0)
print(json.dumps(P, indent=2))

## Inputs, provenance and campaign checks

The requested window is selected by `time_best`, while the printed file names are **close times**. Literal input keys avoid the known unreliable antenna header mapping. This checkpoint requires phase C, synchronized times, RFANT, constant accumulation/integration time and a uniform cadence. It checks the campaign's declared mandatory windows and box-air outages. RFI/comb intervals are deliberately retained for detection rather than removed as whole files.

The archive's `INDEX.md` and `curation/select_files.py` distinguish the exact 8-channel instrumental comb from a walking near-1-MHz transmitter comb. Here 8/4/2 are **pattern hypotheses**, not source identification. A walking comb is not covered by the grid-locked template detector; point/band flags still apply. Source: those local files and `data/README.md`.

In [ ]:
idx = MetadataIndex(CAMPAIGN / 'data')
sel = idx.select()
meta = sel.meta.assign(span=sel.visits(gap_s=600))
spans = meta.groupby('span').agg(start=('time_best','min'), end=('time_best','max'),
                                rows=('file','size'), files=('file','nunique'))
spans[['start','end']] = spans[['start','end']].apply(pd.to_datetime, unit='s', utc=True)
display(spans)
t0 = spans.loc[SPAN, 'start'].timestamp()
t1 = spans.loc[SPAN, 'end'].timestamp()
#window = idx.select(time=(t1 - MINUTES*60, t1))
window = idx.select(time=(t0 + 3600, t0 + 3600 + MINUTES*60))
assert set(window.meta.filter_phase) == {'C'}
assert window.meta.sync_consistent.all(), 'Estimated clock: select a different window.'
assert set(window.meta.rfswitch) == {'RFANT'}, 'Split on calibration switch changes.'
assert window.meta.integration_time.nunique() == 1, 'Split at accumulation changes.'
# Read campaign mask declarations without writing or running its CLI.
import runpy
catalog = runpy.run_path(str(CAMPAIGN / 'curation/select_files.py'))
checks = [(x['name'], x['start'], x['end']) for x in catalog['MANDATORY_MASKS']]
checks += [(x['name'], a, b) for x in catalog['CONDITIONAL_MASKS']
           if x['name'] in ('box-air-outage', 'mux-tuning') for a,b in x.get('windows', [])]
for name, a, b in checks:
    lo, hi = pd.Timestamp(a).timestamp(), pd.Timestamp(b).timestamp()
    assert not window.meta.time_best.between(lo, hi).any(), f'Selection overlaps {name}'
assert not set(window.meta.file) & catalog['CORRUPT_FILES']
bundles = {name: window.load_bundle(key=key, band_mhz=BAND_MHZ, missing='raise')
           for name,key in [('box-gnd','0'), ('box-air','4'), ('cross','04')]}
b = bundles['box-air']
for other in bundles.values():
    assert np.array_equal(other.t, b.t) and np.array_equal(other.freqs_mhz, b.freqs_mhz)
    assert other.meta[['file','row']].equals(b.meta[['file','row']])
t, f = b.t, b.freqs_mhz
D = np.asarray(b.data, float)
G = np.asarray(bundles['box-gnd'].data, float)
X = bundles['cross'].data
integration_s = b.meta.integration_time.to_numpy(float)
assert np.allclose(np.diff(t), integration_s[0], rtol=1e-5)
df_mhz = np.diff(f).mean()
assert np.allclose(np.diff(f), df_mhz)
channel = np.rint(f / df_mhz).astype(int)  # DC-anchored native channels, not cropped indices.
assert np.allclose(channel * df_mhz, f)
sqrt_n = np.sqrt(2 * integration_s[:,None] * df_mhz * 1e6)
domain = (f >= TESTED_BAND_MHZ[0]) & (f < TESTED_BAND_MHZ[1])
valid = np.isfinite(D) & np.isfinite(G) & np.isfinite(X) & (D > 0) & (G > 0) & domain
coherence_z = np.abs(X) / np.sqrt(np.maximum(G,1)*np.maximum(D,1)) * sqrt_n
files = sorted(window.meta.file.unique())
print(window.summary())
print('UTC samples:', pd.to_datetime(t[[0,-1]], unit='s', utc=True).tolist())
print('Phase C; inputs 0, 4, 04. File close-time list:'); print('\n'.join(files))
print('Shape:', D.shape, 'integration s:', integration_s[0], 'channel MHz:', df_mhz)
print('Nonpositive auto counts:', {'ground': int((G<=0).sum()), 'air': int((D<=0).sum())})

def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def git_state(path):
    def git(*args): return subprocess.check_output(['git','-C',str(path),*args], text=True).strip()
    return dict(commit=git('rev-parse','HEAD'), branch=git('branch','--show-current'),
                dirty=bool(git('status','--porcelain','--untracked-files=normal')))
provenance = dict(selection=dict(span=SPAN, minutes=MINUTES, start_utc=str(pd.to_datetime(t[0],unit='s',utc=True)),
                                end_utc=str(pd.to_datetime(t[-1],unit='s',utc=True))),
                  inputs={'box-gnd':'0','box-air':'4','cross':'04'},
                  analysis=git_state(ROOT/'data-analysis'), campaign=git_state(CAMPAIGN),
                  v0_sha256=sha(NOTEBOOK_DIR/'rfi_dev_v0.ipynb'),
                  notebook_code_sha256=hashlib.sha256('\n'.join(''.join(c['source']) for c in json.loads((NOTEBOOK_DIR/'rfi_dev_v1.ipynb').read_text())['cells'] if c['cell_type']=='code').encode()).hexdigest(),
                  mask_catalog_sha256=sha(CAMPAIGN/'curation/select_files.py'),
                  raw_sha256={fn:sha(CAMPAIGN/'data'/fn) for fn in files},
                  installed={p:version(p) for p in ['numpy','scipy','hera_filters','eigsep_data']},
                  parameters=P)
import eigsep_data, eigsep_data.bundle, hera_filters.dspec
provenance['installed_source'] = {m.__name__: dict(path=m.__file__, sha256=sha(m.__file__))
                                 for m in [eigsep_data, eigsep_data.bundle, hera_filters.dspec]}
display(provenance)

## v0 baseline and a matrix-free joint fit

`model(t,f) = A_t C A_f.T`. With a 2-D keep mask `W`, minimize

`sum(W * ((data - model) / scale_f)**2) + ridge * ||C_whitened||**2`.

`scale_f` is a fixed robust spectrum used only for numerical/relative weighting. QR of `A_f / scale_f` changes coordinates, **not the physical DPSS model space**. Each CG operation projects in frequency and time; it never builds the full Kronecker matrix. Preconditioning uses the time and frequency marginals of `W`. Its two Cholesky factorizations have dimensions equal to the basis ranks, independent of the number of distinct row masks. The tiny ridge stabilizes unconstrained directions; extrapolation remains uncertain.

The v0 comparison preserves its `zscore0 > 20` and separable mask logic, including its NaN comparison behavior for negative autos. V1 explicitly excludes those invalid samples. V0's separable training mask and a v0 residual-based candidate mask are reported separately so the retention comparison is not misleading.

In [ ]:
def bases(times, freqs):
    af = dpss_operator(freqs*1e6, [0], [P['freq_halfwidth_s']],
                       eigenval_cutoff=[P['eigenvalue_cutoff']])[0].real
    at = dpss_operator(times-times[0], [0], [P['time_halfwidth_hz']],
                       eigenval_cutoff=[P['eigenvalue_cutoff']])[0].real
    return at, af

A_t, A_f = bases(t, f)
with np.errstate(invalid='ignore', divide='ignore'):
    v0_coherence = np.abs(X) / (np.sqrt(G)*np.sqrt(D)).clip(1,np.inf) * sqrt_n
v0_point_keep = ~(v0_coherence > P['coherence_cut'])
v0_mf = v0_point_keep.mean(axis=0) > .9
v0_mt = np.all(v0_point_keep[:,v0_mf], axis=1)

def baseline(data, mf, mt):
    start = time.perf_counter()
    ff = fit_solution_matrix(mf.astype(float), A_f)
    ft = fit_solution_matrix(mt.astype(float), A_t)
    model = A_t @ (ft @ ((A_f @ (ff @ data.T)).T * mt[:,None]))
    return model, dict(seconds=time.perf_counter()-start, factorizations=2,
                       frequency_rank=A_f.shape[1], time_rank=A_t.shape[1])
model_v0, cost_v0 = baseline(D, v0_mf, v0_mt)
v0_fit_keep = v0_mt[:,None] & v0_mf[None,:]
print('v0:', cost_v0, 'frequency channels:', v0_mf.sum(), 'time rows:', v0_mt.sum())

class TensorFit:
    def __init__(self, times, freqs, data):
        at, af = bases(times, freqs)
        self.qt = np.linalg.qr(at)[0]
        self.scale = np.maximum(median_filter(np.median(np.maximum(data,1),axis=0),size=31),1e4)
        self.qf = np.linalg.qr(af/self.scale[:,None])[0]
        self.shape = (self.qt.shape[1], self.qf.shape[1])
        self.history = []

    def solve(self, data, keep):
        start = time.perf_counter()
        qt,qf = self.qt,self.qf
        w = np.asarray(keep,float)
        assert w.sum() > np.prod(self.shape), 'Insufficient fitting samples.'
        y = np.where(keep, data/self.scale, 0.)  # Invalid values must not enter 0*NaN.
        rhs = qt.T @ y @ qf
        ridge = P['ridge']
        def matvec(a):
            a = a.reshape(self.shape)
            return (qt.T @ (w*(qt @ a @ qf.T)) @ qf + ridge*a).ravel()
        op = LinearOperator((rhs.size,rhs.size),matvec=matvec,dtype=float)
        ct = cho_factor(qt.T @ (w.mean(1)[:,None]*qt)/w.mean() + ridge*np.eye(self.shape[0]))
        cf = cho_factor(qf.T @ (w.mean(0)[:,None]*qf) + ridge*np.eye(self.shape[1]))
        def precondition(a):
            return cho_solve(cf,cho_solve(ct,a.reshape(self.shape)).T).T.ravel()
        pre = LinearOperator(op.shape,matvec=precondition,dtype=float)
        iterations = []
        coeff,info = cg(op,rhs.ravel(),M=pre,rtol=P['cg_rtol'],maxiter=P['cg_maxiter'],
                        callback=lambda _:iterations.append(1))
        rel = np.linalg.norm(matvec(coeff)-rhs.ravel()) / max(np.linalg.norm(rhs),1e-30)
        record = dict(seconds=time.perf_counter()-start, iterations=len(iterations),
                      info=int(info), relative_normal_residual=float(rel), factorizations=2,
                      kept_fraction=float(w.mean()))
        self.history.append(record)
        if info != 0: raise RuntimeError(f'CG did not converge: {record}')
        return (qt @ coeff.reshape(self.shape) @ qf.T)*self.scale

def residual_z(data, model, normalization):
    return np.where(model > 0, (data/np.maximum(model,1)-1)*normalization, np.inf)

def robust_model(data, eligible, solver, normalization):
    keep = eligible.copy()
    trace = []
    for iteration in range(P['robust_rounds']):
        model = solver.solve(data, keep)
        z = residual_z(data,model,normalization)
        # Always reconsider eligible lower samples; no monotone loss of initial FM gaps.
        updated = eligible & np.isfinite(z) & (z < P['fit_clip'])
        change = np.mean(updated != keep)
        trace.append(dict(round=iteration, changed_fraction=change, kept=updated.mean()))
        keep = updated
        if change < P['mask_change_tol']: break
    # Ensure the returned model corresponds exactly to the returned training mask.
    model = solver.solve(data,keep)
    return model,keep,pd.DataFrame(trace)

# Independent mathematical check: dense weighted least squares versus tensor CG.
rng = np.random.default_rng(912)
s = TensorFit.__new__(TensorFit)
s.qt = np.linalg.qr(rng.normal(size=(31,4)))[0]
s.qf = np.linalg.qr(rng.normal(size=(27,5)))[0]
s.scale = np.ones(27); s.shape=(4,5); s.history=[]
y = rng.normal(size=(31,27)); w = rng.random(y.shape) > .35
pred = s.solve(y,w)
B = np.einsum('ti,fj->tfij',s.qt,s.qf).reshape(y.size,-1)
coef = np.linalg.solve(B[w.ravel()].T@B[w.ravel()] + P['ridge']*np.eye(20),
                       B[w.ravel()].T@y.ravel()[w.ravel()])
np.testing.assert_allclose(pred,(B@coef).reshape(y.shape),atol=1e-7)
print('Dense solver agreement passed; DPSS ranks:', A_t.shape[1], A_f.shape[1])

## Band and comb events — no rolling time average

The point flag uses positive auto residuals, cross coherence, invalid counts and negative/model-failure residuals. Band scores use signed, clipped residuals centered/scaled robustly over time, aggregated within each supplied band and in 8-channel tiles within it. A group is excluded at an integration only when its pooled score and fraction of affected channels both pass. Detected comb teeth are excluded from band pooling so harmonics do not automatically veto the good channels between teeth. No score is averaged forward in time. One integration is guarded on either side **after** detection; latency tests disable that guard.

Comb evidence uses local spectral curvature of auto residuals, pooled by **native channel modulo 8**, outside FM to avoid bright stations dominating the template. At least 16 teeth, a minimum occupied fraction, and support in three frequency regions are required. The 4-channel hypothesis requires both constituent mod-8 phases; the 2-channel hypothesis requires all four. This prevents one strong 8-channel comb from being relabeled as contamination on every second channel. All phases are searched. A detected phase flags its teeth across the tested band, including FM. Persistent combs can trigger through spectral contrast even when temporal centering removes their mean.

Defaults are deliberately conservative, but these thresholds are heuristic, not a guaranteed false-alarm probability. Broad smooth interference can be degenerate with the continuum; persistent uncorrelated RFI can bias the model. Masks distinguish reasons below.

In [ ]:
def temporal_center_scale(z, usable):
    a = np.where(usable & np.isfinite(z),z,np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        center = np.nanmedian(a,axis=0)
        scale = 1.4826*np.nanmedian(np.abs(a-center),axis=0)
    return np.nan_to_num(center), np.maximum(np.nan_to_num(scale,nan=1.),1.)

def detect(data, model, coh, valid, normalization, freqs, channels, guard=True):
    z = residual_z(data,model,normalization)
    point_threshold = np.full(len(freqs),P['other_point_cut'])
    for lo,hi in REPORT_BANDS.values():
        point_threshold[(freqs>=lo)&(freqs<hi)] = P['point_cut']
    reasons = {'invalid_or_outside_domain': ~valid,
               'coherence': coh > P['coherence_cut'],
               'positive_excess': z > point_threshold,
               'negative_or_model_failure': (z < -P['negative_cut']) | ~np.isfinite(z)}
    center,scale = temporal_center_scale(z, valid & (coh<P['coherence_cut']) & (z<P['fit_clip']))
    u = np.nan_to_num((z-center)/scale,nan=0.,posinf=0.,neginf=0.)

    finite_z = np.where(np.isfinite(z),z,0.)
    curvature = finite_z - .5*(np.roll(finite_z,1,axis=1)+np.roll(finite_z,-1,axis=1))
    curvature = np.nan_to_num(curvature,nan=0.,posinf=0.,neginf=0.) / np.sqrt(1.5)
    # Never use wraparound neighbours or the untested low-count edge as evidence.
    evidence = valid & np.isfinite(z)
    evidence &= np.roll(evidence,1,axis=1) & np.roll(evidence,-1,axis=1)
    evidence[:,[0,-1]] = False
    evidence &= ~((freqs>=88)&(freqs<108))[None,:]
    _,cs = temporal_center_scale(curvature,evidence)
    tooth = curvature/cs
    comb_scores = np.zeros((len(data),8)); phase_on=np.zeros_like(comb_scores,dtype=bool)
    regions = [(35,108),(108,174),(174,235)]
    for phase in range(8):
        j = channels%8 == phase
        good = evidence[:,j]; count=good.sum(1)
        score=(np.clip(tooth[:,j],-3,8)*good).sum(1)/np.sqrt(np.maximum(count,1))
        hits=(tooth[:,j]>P['comb_tooth_cut'])&good
        coverage=np.ones(len(data),dtype=bool)
        for lo,hi in regions:
            region=(freqs[j]>=lo)&(freqs[j]<hi)
            coverage &= hits[:,region].sum(1)>=2
        on=(score>P['comb_cut']) & (hits.sum(1)>=np.maximum(4,P['comb_fraction']*count))
        on &= coverage & (count>=P['comb_min_teeth'])
        comb_scores[:,phase]=score; phase_on[:,phase]=on
    comb_events={}
    for spacing in (8,4,2):
        for phase in range(spacing):
            comb_events[f'{spacing}:{phase}']=np.all(phase_on[:,phase::spacing],axis=1)
    comb = phase_on[:,channels%8]
    reasons['comb_event']=comb
    bands = np.zeros_like(valid)
    band_scores,band_events = {},{}
    for label,(lo,hi) in REPORT_BANDS.items():
        js = np.flatnonzero((freqs>=lo)&(freqs<hi))
        groups = [(label,js)] + [(f'{label}/tile{k}',js[k:k+P['tile_channels']])
                               for k in range(0,len(js),P['tile_channels'])]
        for name,j in groups:
            if len(j)<3: continue
            good = valid[:,j] & ~comb[:,j]
            count = good.sum(1)
            raw = (np.clip(u[:,j],-3,6)*good).sum(1)/np.sqrt(np.maximum(count,1))
            med = np.median(raw); sigma=max(1.,1.4826*np.median(np.abs(raw-med)))
            score = (raw-med)/sigma
            fraction = ((u[:,j]>P['group_cell_cut'])&good).sum(1)/np.maximum(count,1)
            on = (score>P['group_cut']) & (fraction>=P['group_fraction']) & (count>=3)
            bands[np.ix_(on,j)] = True
            band_scores[name],band_events[name] = score,on
    reasons['band_event'] = bands
    unguarded=np.logical_or.reduce(list(reasons.values()))
    if guard:
        expanded=maximum_filter(unguarded,size=(2*P['time_guard']+1,2*P['frequency_guard']+1),mode='constant')
        reasons['guard']=expanded & ~unguarded
    flag=np.logical_or.reduce(list(reasons.values()))
    return dict(flag=flag,z=z,reasons=reasons,band_scores=band_scores,band_events=band_events,
                comb_scores=comb_scores,comb_events=comb_events,phase_on=phase_on)

def pipeline(data,coh,valid,normalization,times,freqs,channels):
    start=time.perf_counter()
    solver=TensorFit(times,freqs,data)
    eligible=valid & np.isfinite(coh) & (coh<=P['coherence_cut'])
    initial,keep,trace=robust_model(data,eligible,solver,normalization)
    first=detect(data,initial,coh,valid,normalization,freqs,channels)
    # Exclude detected events from a final fit. Keep lower samples during initialization,
    # but exclude clear negative failures now that the continuum is estimated.
    fit_keep=keep & ~first['flag']
    model=solver.solve(data,fit_keep)
    result=detect(data,model,coh,valid,normalization,freqs,channels)
    # Preserve earlier evidence; do not silently unflag a sample after refitting.
    result['reasons']['previous_pass']=first['flag'] & ~result['flag']
    result['flag'] |= first['flag']
    result.update(model=model,fit_keep=fit_keep,solver=solver,trace=trace,
                  seconds=time.perf_counter()-start)
    return result

result=pipeline(D,coherence_z,valid,sqrt_n,t,f,channel)
model_v1=result['model']; flag_v1=result['flag']; z_v1=result['z']
display(result['trace'])
display(pd.DataFrame(result['solver'].history))
print('Total v1 seconds:',result['seconds'])

## Real-window diagnostics

Fractions below use all integrations within each band. “v0 fit” is the separable training support; “v0 residual candidate” uses the same point thresholds as v1 against v0's model plus coherence/validity checks, without group detection. These answer different questions. Reason counts overlap.

In [ ]:
v0_z=residual_z(D,model_v0,sqrt_n)
v0_threshold=np.full(len(f),P['other_point_cut'])
for lo,hi in REPORT_BANDS.values(): v0_threshold[(f>=lo)&(f<hi)]=P['point_cut']
v0_candidate=valid & (coherence_z<=P['coherence_cut']) & (v0_z<=v0_threshold) & (v0_z>=-P['negative_cut'])
rows=[]
for label,(lo,hi) in REPORT_BANDS.items():
    j=(f>=lo)&(f<hi); kept=~flag_v1[:,j]
    rows.append(dict(band=label,channels=int(j.sum()),v0_fit_fraction=v0_fit_keep[:,j].mean(),
                     v0_residual_candidate=v0_candidate[:,j].mean(),v1_keep_fraction=kept.mean(),
                     v1_fit_fraction=result['fit_keep'][:,j].mean(),
                     v1_kept_z_median=np.median(z_v1[:,j][kept]) if kept.any() else np.nan))
retention=pd.DataFrame(rows).set_index('band');display(retention)
display(pd.Series({k:float(v[:,domain].mean()) for k,v in result['reasons'].items()},name='fraction flagged (overlapping)'))
print('Distinct final fitting masks by time:',len(np.unique(np.packbits(result['fit_keep'],axis=1),axis=0)))
print('v0 fit seconds:',cost_v0['seconds'],'v1 total seconds:',result['seconds'],
      'v1 small factorizations:',sum(x['factorizations'] for x in result['solver'].history))

minutes=(t-t[0])/60
extent=[f[0]-df_mhz/2,f[-1]+df_mhz/2,minutes[-1],minutes[0]]
fig,axs=plt.subplots(2,3,figsize=(16,9),sharex=True,sharey=True)
for ax,a,title in zip(axs[0],[D,model_v0,model_v1],['Air auto / counts','v0 background / counts','v1 background / counts']):
    im=ax.imshow(a,aspect='auto',extent=extent,norm=LogNorm(1e3,1e7),interpolation='nearest');ax.set_title(title)
fig.colorbar(im,ax=list(axs[0]),label='counts',shrink=.8)
for ax,a,title in zip(axs[1],[v0_z,z_v1,np.where(flag_v1,np.nan,z_v1)],['v0 residual z','v1 residual z','v1 retained residual z']):
    im=ax.imshow(a,aspect='auto',extent=extent,cmap='RdBu_r',norm=SymLogNorm(1,vmin=-30,vmax=30),interpolation='nearest');ax.set_title(title)
    ax.set_xlabel('Frequency / MHz')
fig.colorbar(im,ax=list(axs[1]),label='signed z',shrink=.8)
for ax in axs[:,0]: ax.set_ylabel('Minutes from '+str(pd.to_datetime(t[0],unit='s',utc=True)))
plt.show()

fm=(f>=88)&(f<108)
fig,axs=plt.subplots(2,2,figsize=(14,8))
axs[0,0].plot(f[fm],np.median(D,axis=0)[fm],label='data median',color='k')
axs[0,0].plot(f[fm],np.median(model_v0,axis=0)[fm],label='v0')
axs[0,0].plot(f[fm],np.median(model_v1,axis=0)[fm],label='v1')
axs[0,0].set_yscale('log');axs[0,0].set_ylabel('Counts');axs[0,0].legend()
axs[0,1].plot(f[fm],v0_candidate[:,fm].mean(0),label='v0 residual candidate')
axs[0,1].plot(f[fm],(~flag_v1[:,fm]).mean(0),label='v1 retained')
axs[0,1].plot(f[fm],result['fit_keep'][:,fm].mean(0),label='v1 fit support',alpha=.6)
axs[0,1].set_ylabel('Fraction of integrations');axs[0,1].legend()
fm_extent=[f[fm][0]-df_mhz/2,f[fm][-1]+df_mhz/2,minutes[-1],0]
axs[1,0].imshow(flag_v1[:,fm],aspect='auto',extent=fm_extent,cmap='gray_r',interpolation='nearest')
axs[1,0].set_title('FM flag: black = excluded');axs[1,0].set_ylabel('Minutes')
im=axs[1,1].imshow(np.where(flag_v1[:,fm],np.nan,z_v1[:,fm]),aspect='auto',extent=fm_extent,
                   cmap='RdBu_r',vmin=-8,vmax=8,interpolation='nearest')
axs[1,1].set_title('FM retained signed residual');fig.colorbar(im,ax=axs[1,1])
for ax in axs.flat: ax.set_xlabel('MHz')
fig.tight_layout();plt.show()

fig,axs=plt.subplots(2,1,figsize=(13,6),sharex=True)
for name,score in result['band_scores'].items():
    if '/tile' not in name: axs[0].plot(minutes,score,label=name,lw=.8)
axs[0].axhline(P['group_cut'],ls='--',color='k');axs[0].set_ylabel('Band score');axs[0].legend(ncol=4,fontsize=8)
axs[1].plot(minutes,result['comb_scores'],lw=.6)
axs[1].axhline(P['comb_cut'],ls='--',color='k');axs[1].set_ylabel('Mod-8 phase score');axs[1].set_xlabel('Minutes')
fig.tight_layout();plt.show()

## Withheld rows: interpolation rather than in-sample residuals

Hold out every 37th time row plus a central 20-second block from **both** model fits. V1 does not use the withheld air values to select its iterative training mask; its fixed numerical scale is recomputed from training rows only. Cross coherence defines a common evaluation subset (`<10`) independently of the air-model residual. Report all of that subset, without removing high residuals afterward. Coherence is not an RFI truth label; the remaining FM contamination and model bias cannot be separated by this test alone.

This comparison tests the **robust fitting stage before event-mask expansion**, rather than the final full detection/refit pipeline. The full-pipeline interpolation error is tested against known synthetic truth below.

In [ ]:
holdout=(np.arange(len(t))%37==0) | ((t-t[0]>590)&(t-t[0]<610))
h0,cost_h0=baseline(D,v0_mf,v0_mt & ~holdout)
hsolver=TensorFit(t,f,D[~holdout])
# TensorFit's time basis still uses all t; only its scale was estimated on training rows.
h1,hkeep,htrace=robust_model(D,valid & (coherence_z<=P['coherence_cut']) & ~holdout[:,None],hsolver,sqrt_n)
hz0=residual_z(D,h0,sqrt_n);hz1=residual_z(D,h1,sqrt_n)
hrows=[]
for name,(lo,hi) in REPORT_BANDS.items():
    evaluate=holdout[:,None] & valid & (coherence_z<10) & ((f>=lo)&(f<hi))[None,:]
    if not evaluate.any(): continue
    for label,z in [('v0',hz0),('v1',hz1)]:
        q=z[evaluate];med=np.median(q)
        hrows.append(dict(band=name,model=label,n=len(q),median_z=med,
                          mad_z=1.4826*np.median(abs(q-med)),p90_abs_z=np.quantile(abs(q),.9)))
holdout_table=pd.DataFrame(hrows);display(holdout_table)
print('Withheld integrations:',holdout.sum(),'v1 CG solves:',len(hsolver.history))

## Controlled injections and null test

Use a known smooth continuum with independent Gaussian radiometer noise and weak cross coherence, then inject persistent FM lines, a weak band burst, 8/4/2-channel combs, a narrow line and a one-integration broadband burst. The 4-sigma comb teeth are below the 6-sigma point threshold: recovery should come from pooled evidence. The FM line locations leave gaps whose model error and retention can be measured against truth.

This is a deterministic engineering test (fixed seed), not a realistic RFI population or a calibrated sensitivity curve. First-row recall disables guard expansion. A noiseless truth model passed directly to the detector separates detector errors from fitting errors. A second end-to-end run includes fitting contamination and a clean null run measures false flags.

In [ ]:
rng=np.random.default_rng(20260918)
ts=np.linspace(0,1200,512)
ns=np.full((len(ts),1),sqrt_n[0,0])
truth=1e6*(1+.12*np.cos(2*np.pi*f[None,:]/150))*(1+.02*np.cos(2*np.pi*ts[:,None]/1800))
clean=truth*(1+rng.normal(size=truth.shape)/ns)
synthetic_coh=np.abs(rng.normal(size=truth.shape)+1j*rng.normal(size=truth.shape))/np.sqrt(2)
sv=np.broadcast_to(domain,truth.shape).copy()
injected=clean.copy();truth_rfi=np.zeros_like(sv);events=[]
def inject(name,start,stop,cols,sigma):
    j=np.flatnonzero(cols);injected[np.ix_(np.arange(start,stop),j)] += sigma*truth[np.ix_(np.arange(start,stop),j)]/ns[start:stop]
    truth_rfi[np.ix_(np.arange(start,stop),j)]=True
    events.append(dict(name=name,start=start,stop=stop,columns=j,sigma=sigma))
inject('persistent FM lines',0,len(ts),fm & (channel%3==0),50)
for name,a,b,spacing,phase in [('comb8',70,100,8,3),('comb4',170,200,4,1),('comb2',300,330,2,0)]:
    inject(name,a,b,domain & (channel%spacing==phase),4)
inject('band burst 108-116',230,245,(f>=108)&(f<116),4)
inject('narrow line',370,400,channel==int(channel[np.argmin(abs(f-145))]),12)
inject('broadband impulse',430,431,domain,30)

oracle=detect(injected,truth,synthetic_coh,sv,ns,f,channel,guard=False)
synth=pipeline(injected,synthetic_coh,sv,ns,ts,f,channel)
synth_raw=detect(injected,synth['model'],synthetic_coh,sv,ns,f,channel,guard=False)
null=pipeline(clean,synthetic_coh,sv,ns,ts,f,channel)
injection_rows=[]
for event in events:
    a,b,j=event['start'],event['stop'],event['columns']
    onset_clean=~truth_rfi[max(a-1,0),j] if a>0 else np.zeros(len(j),bool)
    injection_rows.append(dict(event=event['name'],sigma=event['sigma'],
        oracle_first_recall=oracle['flag'][a,j].mean(),
        fitted_first_recall=synth_raw['flag'][a,j].mean(),
        fitted_event_recall=synth['flag'][a:b][:,j].mean(),
        previous_row_false_fraction=(synth_raw['flag'][a-1,j][onset_clean].mean() if onset_clean.any() else np.nan)))
injection_table=pd.DataFrame(injection_rows);display(injection_table)
clean_cells=sv & ~truth_rfi
fm_gaps=clean_cells & fm[None,:]
model_error=(synth['model']/truth-1)*ns
synthetic_metrics=dict(
    injected_event_recall=float(synth['flag'][truth_rfi].mean()),
    false_flags_outside_injections=float(synth['flag'][clean_cells].mean()),
    null_false_flags=float(null['flag'][sv].mean()),
    fm_gap_retained=float((~synth['flag'][fm_gaps]).mean()),
    fm_gap_model_bias_sigma=float(np.median(model_error[fm_gaps])),
    fm_gap_model_rmse_sigma=float(np.sqrt(np.mean(model_error[fm_gaps]**2))),
    total_model_rmse_sigma=float(np.sqrt(np.mean(model_error[sv]**2))),
    synthetic_seconds=synth['seconds'], null_seconds=null['seconds'])
display(pd.Series(synthetic_metrics))
# Invariants are hard failures. Performance goals are reported, not hidden by assertions.
assert np.isfinite(synth['model'][:,domain]).all()
assert all(x['info']==0 for run in (result,synth,null) for x in run['solver'].history)
assert np.all(flag_v1[~valid])
fm_holdout = holdout_table[holdout_table.band=='88-108 (FM)'].set_index('model')
checks=dict(solver_convergence=True,
    real_fm_retention_exceeds_v0_residual_candidate=bool(retention.loc['88-108 (FM)','v1_keep_fraction'] > retention.loc['88-108 (FM)','v0_residual_candidate']),
    real_fm_holdout_abs_median_improves=bool(abs(fm_holdout.loc['v1','median_z']) < abs(fm_holdout.loc['v0','median_z'])),
    synthetic_onset_recall_ge_90=bool((injection_table.fitted_first_recall>=.9).all()),
    synthetic_null_false_flags_le_1pct=synthetic_metrics['null_false_flags']<=.01,
    synthetic_fm_gap_retention_ge_90pct=synthetic_metrics['fm_gap_retained']>=.9,
    synthetic_fm_model_rmse_le_1sigma=synthetic_metrics['fm_gap_model_rmse_sigma']<=1.)
display(pd.Series(checks,name='checkpoint acceptance'))
print('Real FM recovery is NOT validated by this checkpoint.' if not all(checks.values()) else 'All checkpoint checks passed; independent real-data validation remains necessary.')

fig,axs=plt.subplots(1,3,figsize=(15,5),sharex=True,sharey=True)
for ax,a,title in zip(axs,[truth_rfi,synth['flag'],model_error],['Injected RFI truth','Recovered flags (with guards)','Background error / noise sigma']):
    im=ax.imshow(a,aspect='auto',extent=[f[0],f[-1],ts[-1]/60,0],interpolation='nearest',
                 cmap='RdBu_r' if 'error' in title else 'gray_r',vmin=-1 if 'error' in title else 0,vmax=1)
    ax.set_title(title);ax.set_xlabel('MHz')
axs[0].set_ylabel('Minutes');fig.colorbar(im,ax=axs[2]);fig.tight_layout();plt.show()

## Measured checkpoint summary

The following values are generated by execution. The real-FM checks are deliberately separate from synthetic acceptance: this checkpoint can establish implementation correctness while failing to establish improved real-data recovery.

In [ ]:
fm_row=retention.loc['88-108 (FM)']
print(f"FM fitting support: v0 {100*fm_row.v0_fit_fraction:.2f}% -> v1 {100*fm_row.v1_fit_fraction:.2f}%")
print(f"FM final candidates: v0 residual rule {100*fm_row.v0_residual_candidate:.2f}% vs v1 {100*fm_row.v1_keep_fraction:.2f}%")
print(f"FM held-out median z: v0 {fm_holdout.loc['v0','median_z']:.2f}, v1 {fm_holdout.loc['v1','median_z']:.2f}")
print(f"Runtime: v0 single fit {cost_v0['seconds']:.3f} s; v1 full pipeline {result['seconds']:.3f} s")
print(f"Synthetic FM-gap retention {100*synthetic_metrics['fm_gap_retained']:.2f}%; model RMSE {synthetic_metrics['fm_gap_model_rmse_sigma']:.3f} noise sigma")
print(f"Synthetic clean-cell collateral flags {100*synthetic_metrics['false_flags_outside_injections']:.2f}% (includes guards); null flags {100*synthetic_metrics['null_false_flags']:.3f}%")
print('The efficient arbitrary-mask fit and injected onset tests work. Real FM model improvement remains unresolved; do not interpret extra fitting support as acceptance.')

## Interpretation, limitations and deferred findings

- Use `flag_v1` for the final boolean exclusion mask and `model_v1` for the smooth air-auto model. `result['fit_keep']` records the actual model-training support; a fitting mask is not interchangeable with the final flag. Model values outside the tested band are extrapolations and must not be used as recovered data.
- Runtime compares a full detection/refit pipeline with v0's single model fit; it does not imply equal work. The table records the actual number of solves, CG iterations, distinct row masks and small factorizations. This avoids per-row inversions but still costs more than one separable fit.
- Real retained residuals are selected by the flagger and cannot demonstrate unbiased recovery. Read the held-out table and synthetic known-truth test together. There is no claim that all retained real FM samples are clean.
- Scores use the full window to estimate background and scale. “First integration” means offline onset localization, **not causal streaming latency**. A streaming implementation would need a frozen/past-only background and its own validation.
- Bright off-grid lines can leak into adjacent channels. The default frequency guard is zero to preserve FM gaps; use `frequency_guard=1` for a more conservative experiment and measure the cost. The one-row time guard deliberately sacrifices some neighboring clean data.
- Tensor DPSS smoothness can absorb slowly varying, broadband interference. Very long events, the walking transmitter comb, fully occupied FM intervals, changing pointing/calibration and irregular cadence need additional tests. The current checks reject unsuitable state/cadence changes rather than silently stitching them.
- Negative auto counts are excluded rather than repaired. Integer wrap repair is a separate data interpretation decision. The thermal z normalization is inherited from v0 and should be empirically calibrated before interpreting numeric thresholds as sigma significance.

**Deferred findings:** verify the physical source and exact spacing of each comb against transmitter logs; run multiple campaign windows spanning comb onset and strong harmonics; calibrate thresholds against independent clean/contaminated labels; measure guard/threshold trade-offs; compare DPSS widths and ridge sensitivity. These are not silently performed as another analysis stage.

## Decision requested

Review the executed retention, held-out, injection and timing results. Decide whether the next milestone should emphasize (a) real comb-onset windows and harmonic alias tests, or (b) independent validation of recovered FM gaps and background bias. Failed acceptance checks are limitations of this candidate, not permission to deploy it.

**STOPPED AT REVIEW GATE — awaiting Aaron's approval.**